# LexIA — Phase 5 : Évaluation RAGAS

Ce notebook retrace l'évaluation du système RAG LexIA avec le framework RAGAS.
Il couvre :
- Les 4 métriques RAGAS et leur signification
- La construction du dataset d'évaluation (20 questions)
- La génération des réponses et l'évaluation
- L'analyse des scores obtenus
- Les limitations et axes d'amélioration

**Scores obtenus** :
- Faithfulness : **0.463**
- Context Precision : **0.631**
- Answer Relevancy : n/a (incompatibilité RAGAS v0.4 / Groq)
- Context Recall : n/a (incompatibilité RAGAS v0.4 / Groq)

## 1. RAGAS — Retrieval Augmented Generation Assessment

### Pourquoi RAGAS et pas ROUGE/BLEU ?

ROUGE et BLEU mesurent la similarité lexicale entre une réponse générée et une référence.
Ils ne capturent pas ce qui compte vraiment pour un RAG :
- Le LLM a-t-il bien utilisé les documents retrieved ?
- Les documents retrieved étaient-ils pertinents ?

RAGAS utilise un **LLM-as-judge** pour évaluer ces dimensions sémantiques.

### Les 4 métriques

| Métrique | Question | Spécifique RAG |
|---|---|---|
| **Faithfulness** | La réponse est-elle fidèle aux chunks retrieved ? | ✅ |
| **Answer Relevancy** | La réponse répond-elle à la question ? | ❌ |
| **Context Recall** | Les chunks contiennent-ils l'info nécessaire ? | ✅ |
| **Context Precision** | Les chunks retrieved sont-ils tous pertinents ? | ✅ |

### LLM-as-judge — principe

RAGAS envoie la question, la réponse et les chunks à un LLM juge qui évalue
chaque métrique. C'est plus précis que les métriques lexicales mais introduit
un biais si le même LLM génère les réponses ET les juge.

## 2. Imports et configuration

In [ ]:
import json
import sys
import os
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from dotenv import load_dotenv

sys.path.insert(0, '/workspaces/lexia')
load_dotenv('/workspaces/lexia/.env')

DATASET_PATH  = '/workspaces/lexia/evaluation/eval_dataset.json'
RESULTS_PATH  = '/workspaces/lexia/evaluation/results/ragas_scores.json'
INTERIM_PATH  = '/workspaces/lexia/evaluation/results/responses_interim.json'

print('Imports OK')

## 3. Dataset d'évaluation — 20 questions

In [ ]:
with open(DATASET_PATH, encoding='utf-8') as f:
    dataset = json.load(f)

print(f'Dataset : {len(dataset)} questions')
print()

# Répartition par style et genre
from collections import Counter
by_style = Counter(d.get('style', 'unknown') for d in dataset)
by_genre = Counter(d.get('genre', 'unknown') for d in dataset)

print('── Par style ──────────────────────────────────')
for style, count in by_style.items():
    print(f'  {style:<12} : {count} questions')

print()
print('── Par genre ──────────────────────────────────')
for genre, count in by_genre.items():
    print(f'  {genre:<12} : {count} questions')

print()
print('── Exemples ───────────────────────────────────')
for d in dataset[:3]:
    print(f'  [{d["style"]:>8} / {d["genre"]:>8}] {d["question"][:60]}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Distribution du dataset d\'évaluation LexIA', fontsize=13)

# Par style
ax1 = axes[0]
styles = list(by_style.keys())
counts_style = list(by_style.values())
bars = ax1.bar(styles, counts_style, color=['#1D9E75', '#7F77DD'], edgecolor='white')
for bar, count in zip(bars, counts_style):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             str(count), ha='center', va='bottom', fontsize=11)
ax1.set_title('Par style de question')
ax1.set_ylabel('Nombre de questions')
ax1.grid(axis='y', alpha=0.3)

# Par genre
ax2 = axes[1]
genres = list(by_genre.keys())
counts_genre = list(by_genre.values())
colors_genre = ['#1D9E75', '#D85A30', '#7F77DD']
wedges, texts, autotexts = ax2.pie(
    counts_genre, labels=genres, colors=colors_genre,
    autopct='%1.0f%%', startangle=90,
    textprops={'fontsize': 10}
)
ax2.set_title('Par genre')

plt.tight_layout()
plt.savefig('/workspaces/lexia/notebooks/eval_dataset_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Réponses générées par LexIA

In [ ]:
# Charge les réponses générées (évite de les régénérer)
with open(INTERIM_PATH, encoding='utf-8') as f:
    responses = json.load(f)

print(f'Réponses chargées : {len(responses)}')
print()

# Statistiques sur les réponses
answered     = [r for r in responses if 'Je n\'ai pas trouvé' not in r['answer']]
not_answered = [r for r in responses if 'Je n\'ai pas trouvé' in r['answer']]

print(f'Réponses trouvées    : {len(answered)}/{len(responses)}')
print(f'Réponses non trouvées: {len(not_answered)}/{len(responses)}')
print()

# Exemple d'une réponse
sample = responses[0]
print(f'Q: {sample["question"]}')
print(f'Chunks: {len(sample["contexts"])}')
print(f'Réponse (début): {sample["answer"][:200]}...')

In [ ]:
# Analyse des chunks retrieved
chunks_counts = [len(r['contexts']) for r in responses]

print('── Chunks retrieved par question ──────────────')
print(f'Moyenne : {sum(chunks_counts)/len(chunks_counts):.1f} chunks')
print(f'Min     : {min(chunks_counts)} chunks')
print(f'Max     : {max(chunks_counts)} chunks')
print()

# Distribution
from collections import Counter
dist = Counter(chunks_counts)
for n, count in sorted(dist.items()):
    print(f'  {n} chunks : {count} questions')

## 5. Scores RAGAS obtenus

In [ ]:
with open(RESULTS_PATH, encoding='utf-8') as f:
    results = json.load(f)

scores = results['scores_globaux']

print('── Scores RAGAS ────────────────────────────────')
for metric, score in scores.items():
    if score is not None:
        bar = '█' * int(score * 20)
        print(f'  {metric:<20} : {score:.3f}  |{bar:<20}|')
    else:
        print(f'  {metric:<20} : n/a   (incompatibilité RAGAS/Groq)')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

metrics = ['Faithfulness', 'Context\nPrecision', 'Answer\nRelevancy', 'Context\nRecall']
values  = [0.463, 0.631, None, None]
colors  = ['#1D9E75', '#1D9E75', '#CCCCCC', '#CCCCCC']

bars = ax.bar(
    range(len(metrics)),
    [v if v is not None else 0 for v in values],
    color=colors, edgecolor='white', width=0.6
)

# Annotations
for i, (bar, val) in enumerate(zip(bars, values)):
    if val is not None:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
    else:
        ax.text(bar.get_x() + bar.get_width()/2, 0.05,
                'n/a\n(Groq)', ha='center', va='bottom', fontsize=9, color='#888')

# Ligne de référence
ax.axhline(y=0.7, color='#D85A30', linestyle='--', linewidth=1.5, label='Seuil cible prod (0.7)')
ax.axhline(y=0.5, color='#F0A030', linestyle='--', linewidth=1, label='Seuil acceptable (0.5)')

ax.set_xticks(range(len(metrics)))
ax.set_xticklabels(metrics, fontsize=10)
ax.set_ylim(0, 1.0)
ax.set_ylabel('Score (0 → 1)')
ax.set_title('Scores RAGAS — LexIA v0.1\n(dataset 10 questions formelles)')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

# Légende couleurs
patch_ok  = mpatches.Patch(color='#1D9E75', label='Mesuré')
patch_na  = mpatches.Patch(color='#CCCCCC', label='Non mesuré (incompatibilité Groq)')
ax.legend(handles=[patch_ok, patch_na], loc='upper right', fontsize=9)

plt.tight_layout()
plt.savefig('/workspaces/lexia/notebooks/ragas_scores.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Interprétation des scores

### Faithfulness : 0.463

Le LLM s'éloigne parfois du contexte fourni en ajoutant des formulations générales
non sourcées : *"il est important de noter"*, *"en général"*, etc.

**Action corrective** : ajout d'une règle 7 dans le prompt :
*"N'ajoute AUCUNE formulation générale qui ne provient pas directement des articles fournis."*

### Context Precision : 0.631

63% des chunks retrieved sont pertinents — attendu pour du dense retrieval sans sparse.
BGE-M3 avec hybrid retrieval (dense+sparse) permettrait d'atteindre ~0.75+.

### Answer Relevancy + Context Recall : n/a

RAGAS v0.4 utilise le paramètre `n>1` pour ces métriques — non supporté par l'API Groq.
Solution production : OpenAI GPT-4o-mini comme LLM évaluateur (~0.10€ pour 20 questions).

### Scores potentiellement sous-estimés

- Llama 3.1 8B (juge) est moins précis que GPT-4o pour le jugement sémantique
- Ground_truth générées par LLM (biais circulaire possible)
- Dataset de 10 questions (trop petit pour des scores stables)

## 7. Limitations et axes d'amélioration

| Limitation | Impact | Solution |
|---|---|---|
| LLM juge = LLM système | Biais circulaire | GPT-4o-mini comme juge indépendant |
| Ground_truth LLM | Scores biaisés | Vérification manuelle sur Légifrance |
| RAGAS v0.4 / Groq | 2 métriques manquantes | Migrer vers OpenAI pour l'éval |
| Dense retrieval seul | Context Precision ~0.63 | BGE-M3 hybrid sur 16 Go+ |
| Dataset 10 questions | Scores instables | 50+ questions minimum en prod |

## 8. Prochaine étape — Phase 6 : Monitoring

```python
# Langfuse trace chaque requête automatiquement
from monitoring.langfuse_client import trace_rag_query

trace_id = trace_rag_query(
    question=question,
    answer=response,
    contexts=contexts,
    latency_ms=latency_ms,
)
# → visible sur cloud.langfuse.com
```